# Playbook 4 · Operations and communication

**Stage:** the memo — and everything it has to be true about. Ownership, the 3am
rollback, rebuild cost, and what downstream consumers are told.


> **Playbook, not walkthrough.** [`exploration.ipynb`](exploration.ipynb) is the
> narrative for a reviewer: what was built and why. These five are operational —
> one per assignment stage, each answering *what does this stage guarantee* and
> *what would it take to run it in production*. They overlap deliberately on
> evidence and not at all on purpose.


## What this stage must guarantee

The memo's claims and the repository's behaviour are the same claims. Every
figure in `README.md`, `MEMO.md` and `RETRIEVAL.md` is generated by
`scripts/evidence.py` and `scripts/retrieval_report.py` rather than typed, so
they cannot drift apart silently.

That is a process decision, not a documentation style, and it is why the numbers
in the memo can be trusted without re-deriving them.

In [1]:
from __future__ import annotations

import datetime as dt
import tempfile
import textwrap
from pathlib import Path

from forecast_spine import coverage, fixtures, gates, normalize, pipeline, seasonal_naive

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW = REPO / "data" / "raw"
SQL = REPO / "sql" / "asof_join.sql"

# The assignment window. Actuals for operating day D publish on D+1, so the
# processing date is one day past the last target day.
PROCESSING_DATE = dt.date(2026, 3, 24)
WINDOW_START, WINDOW_END = dt.date(2026, 2, 22), dt.date(2026, 3, 23)
PUBLICATION_START = dt.date(2026, 2, 21)

LIVE = any(RAW.glob("load_forecast/*_csv.zip"))
raw_root = RAW if LIVE else fixtures.build("pass", Path(tempfile.mkdtemp()))
if not LIVE:
    print("LIVE VINTAGES NOT FOUND -- using synthetic fixtures.\n"
          "Structure is preserved; scale and revision behaviour are not.\n")


def build():
    """Build a throwaway warehouse. Never touches data/warehouse/.

    A scratch database keeps this notebook runnable while something else holds
    the committed one -- DuckDB is single-writer, and a SQL client with an open
    connection is enough to block it.
    """
    if LIVE:
        context = pipeline.build_context(
            PROCESSING_DATE, raw_root, window_start=WINDOW_START, window_end=WINDOW_END
        )
    else:
        context = pipeline.build_context(
            fixtures.processing_date_for("pass"), raw_root, window_days=1
        )
    con = pipeline.connect(Path(tempfile.mkdtemp()) / "playbook.duckdb")
    pipeline.load(con, context)
    pipeline.build_evaluation_dataset(con, context, SQL)
    return con, context

## Run it — the numbers the memo quotes

In [2]:
con, context = build()
readiness = gates.data_readiness(con, context)
naive = seasonal_naive.evaluate(con, "naive")
ercot_model = seasonal_naive.evaluate(con, "ercot")

first_day, last_day = context.window_operating_dates
print(f"run_id             {context.run_id}")
print(f"processing date    {context.processing_date}  "
      f"(visibility cutoff {context.processing_ts_utc:%Y-%m-%d %H:%M} UTC)")
print(f"evaluation window  {first_day} .. {last_day}")
print(f"source files       {readiness.metrics['source_files']:,}")
print(f"source rows        {readiness.metrics['raw_source_rows']:,}")
print(f"evaluation rows    {readiness.metrics['evaluation_rows']:,}")
print()
print(f"seasonal-naive     WAPE {naive.wape_pct:.2f}%  bias {naive.bias_mw:+,.0f} MW")
print(f"ERCOT in-use model WAPE {ercot_model.wape_pct:.2f}%")
print()
print("Regenerate everything the prose quotes:")
print("  uv run python scripts/evidence.py")
print("  uv run python scripts/retrieval_report.py --verify")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

run_id             ae80e7096413957c6de9fafe9058c20b
processing date    2026-03-24  (visibility cutoff 2026-03-25 05:00 UTC)
evaluation window  2026-02-22 .. 2026-03-23
source files       775
source rows        1,132,967
evaluation rows    5,752

seasonal-naive     WAPE 8.65%  bias -68 MW
ERCOT in-use model WAPE 3.39%

Regenerate everything the prose quotes:
  uv run python scripts/evidence.py
  uv run python scripts/retrieval_report.py --verify


## What would make me roll this back at 3am

The brief asks for **observable symptoms an on-call operator could see**, not
model metrics. An operator at 3am does not know what WAPE is and should not have
to.

| Symptom | What it probably means | Immediate action |
| --- | --- | --- |
| Evaluation published with a `run_id` that is not in the release log | The gate was bypassed, or a manual run escaped | Withdraw the artifact. Find who ran it. |
| Two runs for the same processing date with **different** `run_id`s | Inputs changed underneath a supposedly frozen date | Freeze publication. Diff the source manifests. |
| Forecast error suddenly *much better* than its trailing weeks | **Leakage**, not improvement | Roll back. Compare as-of WAPE against latest-vintage WAPE. |
| Row counts up sharply with no coverage change | Grain broken by a join | Roll back. Check `count(*)` against zones × target hours. |
| Quarantine rate drops to zero after being non-zero | The detector broke, not the source | Do not celebrate. Check `schema_fingerprint`. |
| Evaluation published while acquisition is stale | Scoring against a partial window | Withdraw. See Playbook 0. |

The third row is the one worth rehearsing. **A sudden improvement is the alarm.**
Every other system trains operators to treat improvement as good news; here it is
the single most likely signature of a correctness regression.

## Rebuild cost, and what to cache

| Stage | Cost, this window | Scales with |
| --- | --- | --- |
| Acquisition | ~30 min at 28 req/min | number of publications; rate limit is the floor |
| Normalize + load | **~40s** for 775 files, 1.13M rows | total source lines |
| As-of join | ~2s | vintages × target hours |
| Gates | < 1s | evaluation rows |
| **Full rebuild from disk** | **42.6s** | |

Retrieval dominates wall-clock; normalization dominates rebuild. **Cache
normalized rows keyed by `content_sha256`** — files are immutable, so a rebuild
that re-reads the same bytes can skip parsing entirely. Do not cache the as-of
join: it is cheap, and a stale one is exactly the failure this system exists to
prevent.

## The dominant uncertainty

If one thing here is wrong, the rest does not matter: **the MIS filename
timestamp is the moment a vintage became available.**

Every cutoff is computed from it. If ERCOT's real posting time differs
materially from that stamp, every selection is off by that difference — in a
direction undetectable from the files themselves.

**What I would measure first:** poll the MIS listing continuously for a week,
record the first instant each filename is observable, compare against the stamp.
That converts an assumption into a distribution, and it is the one measurement I
would want before trusting any of this in production.

## Open issues

| Issue | Status | What would resolve it |
| --- | --- | --- |
| Thresholds calibrated on September, applied to March; all three trip | Open, deliberate | Per-month rolling baseline. Not built. |
| Zero quarantined rows in 1.13M real rows | Open, stated | A naturally occurring source-shape change, or a longer window |
| 2026-03-06's five missing publications: ours or ERCOT's? | **Resolved — ERCOT's** | Done. `--verify`: archive lists 19, we hold 19 — never published. |
| September vintages irreplaceable — MIS retains ~7 days, gitignored | Accepted | Nothing. Stated so no reviewer wastes time. |
| Uniform mild bias passes an absolute-error gate | Open, compensated | Gate on signed bias runs; today only monitored |
| Actuals assumed never revised after first publication | Assumption | Measured true over this window; not guaranteed |

## Ownership, if this were a real service

| Concern | Owner | Cadence |
| --- | --- | --- |
| Acquisition freshness | data platform on-call | continuous, paged |
| Gate verdicts, correctness class | data platform on-call | paged |
| Gate verdicts, model class | forecasting | next business day |
| Threshold review | forecasting + platform, jointly | quarterly, and **never** in the same change that observed a failure |
| Source contract with ERCOT | data platform | on schema drift |

**What downstream consumers are told.** Every published evaluation carries its
`run_id`, its processing date, and its gate verdicts. A consumer who cannot name
the `run_id` behind a number they are quoting is quoting a number nobody can
reproduce — and that, not a bad forecast, is the failure mode that costs money
quietly.

## Where this lives

`MEMO.md` · `README.md` · `RETRIEVAL.md` · `scripts/evidence.py` ·
`scripts/retrieval_report.py`